# ALIA Prompting Smoke Test


In [ ]:
from __future__ import annotations

import os
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

ARTIFACT_ROOT = Path.cwd() / 'artifacts'
TEST_ROOT = ARTIFACT_ROOT / 'carolyn_alia_test'
OUTPUT_ROOT = TEST_ROOT / 'generated'
for p in [TEST_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

ALIA_REPO_URL = 'https://github.com/lisadunlap/ALIA.git'
ALIA_REPO_PATH = Path('ALIA')

if not torch.cuda.is_available():
    print('Warning: the ALIA editing scripts are CUDA-first and may fail without a GPU.')


In [ ]:
def git(*args: str) -> subprocess.CompletedProcess[str]:
    return subprocess.run(['git', *args], capture_output=True, text=True, check=False)

remote_check = git('ls-remote', ALIA_REPO_URL, 'HEAD')
print('ALIA remote status:', remote_check.returncode)
print((remote_check.stdout or remote_check.stderr).splitlines()[:3])

if not ALIA_REPO_PATH.exists():
    clone_check = git('clone', '--depth', '1', ALIA_REPO_URL, str(ALIA_REPO_PATH))
    print('ALIA clone status:', clone_check.returncode)
    if clone_check.returncode != 0:
        print((clone_check.stdout or clone_check.stderr).splitlines()[:10])

if ALIA_REPO_PATH.exists():
    head_check = git('-C', str(ALIA_REPO_PATH), 'rev-parse', 'HEAD')
    print('ALIA local HEAD:', head_check.returncode, head_check.stdout.strip())

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from alia_speciesnet_helpers import ALIA_EDIT_SPECS, build_alia_prompt, run_alia_img2img_example

print('ALIA edit specs:', [spec.name for spec in ALIA_EDIT_SPECS])


In [ ]:
sample_image = ALIA_REPO_PATH / 'ex_imgs' / 'giraffe.png'
if not sample_image.exists():
    raise FileNotFoundError(f'Missing sample image: {sample_image}')

species_lookup = {'giraffe': 'Giraffa camelopardalis'}
records = []
fig, axes = plt.subplots(1, len(ALIA_EDIT_SPECS) + 1, figsize=(5 * (len(ALIA_EDIT_SPECS) + 1), 5))
original = Image.open(sample_image).convert('RGB')
axes[0].imshow(original)
axes[0].set_title('Original\ngiraffe')
axes[0].axis('off')

for idx, spec in enumerate(ALIA_EDIT_SPECS, start=1):
    prompt = build_alia_prompt(spec.name, 'giraffe', species_lookup=species_lookup, seed=RANDOM_SEED + idx)
    print(f'Running {spec.name}: {prompt}')
    generated_paths = run_alia_img2img_example(
        repo_path=ALIA_REPO_PATH,
        image_path=sample_image,
        prompt=prompt,
        output_dir=OUTPUT_ROOT / spec.name / 'giraffe',
        n=1,
        strength=0.55 if spec.name == 'fine_grained' else 0.65,
        guidance=7.5,
    )
    edited_path = generated_paths[0]
    edited = Image.open(edited_path).convert('RGB')
    axes[idx].imshow(edited)
    axes[idx].set_title(spec.name)
    axes[idx].axis('off')
    records.append({'method': spec.name, 'prompt': prompt, 'path': str(edited_path)})

plt.tight_layout()
plt.show()

alia_outputs = pd.DataFrame(records)
display(alia_outputs)
alia_outputs.to_csv(TEST_ROOT / 'alia_prompt_smoke_test.csv', index=False)
